# 🎙️ Piper Vietnamese Voice Training Studio (Advanced Dataset & Drive Edition)\n\nCông cụ **Huấn luyện Mô hình Giọng nói Tiếng Việt VITS** trên **Google Colab GPU T4** hỗ trợ:\n1. 📁 **Nạp Dataset trực tiếp từ Google Drive:** Có sẵn danh sách Dropdown chọn nhanh các bộ dataset (kevinha 247, 819, 2625 câu) hoặc nhập đường dẫn tùy chỉnh.\n2. 🔀 **Random Shuffle:** Tự động xáo trộn ngẫu nhiên dữ liệu câu thoại (không ảnh hưởng file gốc trên Drive).\n3. 🎯 **Tùy chọn số lượng câu:** Chọn số lượng câu muốn train (ví dụ chỉ lấy 50, 100 câu đầu sau khi shuffle).\n4. ♻️ **Load Checkpoint từ Drive:** Tiếp tục huấn luyện từ checkpoint cá nhân đã lưu trên Google Drive hoặc từ Base tiếng Việt gốc.\n5. 💾 **Lưu Checkpoint mới vào Google Drive:** Tự động tạo thư mục theo Voice ID tại MyDrive/Piper_Backup/checkpoints/<voice_name>.\n6. ⚡ **Xuất ONNX & ONNX.JSON:** Tải trực tiếp mô hình nén siêu nhẹ để dùng ngay trong Piper TTS Studio.\n

In [ ]:
# @title 📁 Bước 0: Kết Nối (Mount) Google Drive { display-mode: "form" }
from google.colab import drive
import os

if not os.path.exists('/content/drive/MyDrive'):
    print('Đang kết nối Google Drive...', flush=True)
    drive.mount('/content/drive')
    print('✅ Google Drive đã sẵn sàng!')
else:
    print('✅ Google Drive đã được kết nối tại: /content/drive/MyDrive')



In [ ]:
# @title 🚀 Bước 1: Tạo Python 3.11 riêng, cài Piper và tải Checkpoint Tiếng Việt { display-mode: "form" }
import os, sys, shutil, subprocess, time, urllib.request
from pathlib import Path

SETUP_LOG = Path('/content/piper_setup.log')
SETUP_LOG.write_text('', encoding='utf-8')

def run_setup_step(label, command, **kwargs):
    print(f'\n-> {label}', flush=True)
    with SETUP_LOG.open('a', encoding='utf-8') as log:
        log.write(f'\n$ {command!r}\n')
        log.flush()
        process = subprocess.Popen(
            command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, **kwargs
        )
        if process.stdout is None:
            raise RuntimeError(f'Không đọc được log của bước: {label}')
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

print('======================================================================', flush=True)
print('🎙️ KHỞI TẠO PIPER TRAINING ENGINE CHO GOOGLE COLAB GPU T4...', flush=True)
print('======================================================================', flush=True)

def download_with_progress(url, dest_path, desc='Đang tải file'):
    dest = Path(dest_path)
    if dest.exists() and dest.stat().st_size > 1000000:
        print(f'✅ {desc}: Đã có sẵn ({dest.stat().st_size / (1024*1024):.1f} MB)', flush=True)
        return str(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix('.download_tmp')
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    try:
        with urllib.request.urlopen(req) as resp, open(tmp, 'wb') as f:
            total_size = int(resp.headers.get('Content-Length', 0))
            downloaded = 0
            start_time = time.time()
            last_print = 0
            while True:
                chunk = resp.read(2 * 1024 * 1024)
                if not chunk: break
                f.write(chunk)
                downloaded += len(chunk)
                now = time.time()
                if now - last_print > 0.4 or downloaded == total_size:
                    last_print = now
                    elapsed = max(0.1, now - start_time)
                    speed = (downloaded / (1024 * 1024)) / elapsed
                    pct = int((downloaded / total_size) * 100) if total_size > 0 else 0
                    mb_down = downloaded / (1024 * 1024)
                    mb_tot = total_size / (1024 * 1024)
                    sys.stdout.write(f'\r-> {desc}: [{pct}%] ({mb_down:.1f} MB / {mb_tot:.1f} MB) - {speed:.1f} MB/s')
                    sys.stdout.flush()
        print('', flush=True)
        tmp.rename(dest)
        print(f'✅ {desc}: Tải hoàn tất!', flush=True)
        return str(dest)
    except Exception as e:
        if tmp.exists(): tmp.unlink()
        print(f'\n⚠️ Lỗi khi tải {desc}: {e}', flush=True)
        return None

# 1. Gói hệ thống eSpeak-NG
run_setup_step('[1/4] Cập nhật danh sách gói hệ thống', ['apt-get', 'update'])
run_setup_step('[1/4] Cài eSpeak-NG, build tools và FFmpeg', ['apt-get', 'install', '-y', 'build-essential', 'espeak-ng', 'libespeak-ng-dev', 'ffmpeg'])

# 2. Tạo môi trường Python 3.11 riêng cho Piper legacy
print('\n-> [2/4] Tạo môi trường Python 3.11 riêng cho Piper legacy...', flush=True)
uv_exe = shutil.which('uv')
if not uv_exe:
    run_setup_step('Cài uv', [sys.executable, '-m', 'pip', 'install', 'uv'])
    uv_exe = shutil.which('uv')
if not uv_exe:
    raise FileNotFoundError('Không tìm thấy lệnh uv sau khi cài đặt.')
run_setup_step('Tải Python 3.11', [uv_exe, 'python', 'install', '3.11'])
PIPER_ENV = Path('/content/piper_py311')
if not (PIPER_ENV / 'bin' / 'python').is_file():
    run_setup_step('Tạo virtualenv Python 3.11', [uv_exe, 'venv', '--python', '3.11', str(PIPER_ENV)])
PIPER_PYTHON = str(PIPER_ENV / 'bin' / 'python')

run_setup_step('Cài PyTorch 2.6.0 CUDA 12.4 cho T4', [
    uv_exe, 'pip', 'install', '--python', PIPER_PYTHON, '--no-progress',
    '--index-url', 'https://download.pytorch.org/whl/cu124', 'torch==2.6.0'
])

print('-> Cài Piper-Phonemize và thư viện Python...', flush=True)
try:
    run_setup_step('Cài piper-phonemize', [uv_exe, 'pip', 'install', '--python', PIPER_PYTHON, '--only-binary=:all:', 'piper-phonemize==1.1.0'])
except subprocess.CalledProcessError as e:
    raise RuntimeError('Cài piper-phonemize thất bại. Xem log ở trên hoặc /content/piper_setup.log.') from e
run_setup_step('Cài thư viện huấn luyện và Gradio', [
    uv_exe, 'pip', 'install', '--python', PIPER_PYTHON, '--no-progress', 'setuptools==80.9.0', 'numpy==2.0.2', 'gradio', 'onnx', 'soundfile', 'pandas',
    'pytorch-lightning==1.9.5', 'tensorboard', 'cython>=0.29,<1', 'librosa', 'onnxruntime'
])

# 3. Cài đặt piper_train
print('\n-> [3/4] Cài đặt Piper VITS Training Engine (piper_train)...', flush=True)
if not os.path.exists('piper'):
    run_setup_step('Clone mã nguồn Piper', ['git', 'clone', '--depth', '1', 'https://github.com/rhasspy/piper.git'])

piper_src = os.path.abspath('piper/src/python')
piper_env = os.environ.copy()
piper_env['PATH'] = str(PIPER_ENV / 'bin') + os.pathsep + piper_env.get('PATH', '')
piper_env['PYTHONPATH'] = piper_src
if os.path.exists(piper_src):
    run_setup_step('Cài piper_train', [uv_exe, 'pip', 'install', '--python', PIPER_PYTHON, '--no-deps', '-e', '.'], cwd=piper_src, env=piper_env)
    build_script = os.path.join(piper_src, 'build_monotonic_align.sh')
    run_setup_step('Biên dịch monotonic_align', ['bash', build_script], cwd=piper_src, env=piper_env)
else:
    raise FileNotFoundError(f'Không tìm thấy mã nguồn piper_train: {piper_src}')

# 4. Tải Checkpoint Tiếng Việt gốc để fine-tune siêu tốc
print('-> [4/4] Tải checkpoint và kiểm tra môi trường...', flush=True)
ckpt_url = 'https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/vi/vi_VN/vais1000/medium/epoch=4769-step=919580.ckpt'
cfg_url = 'https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/vi/vi_VN/vais1000/medium/config.json'
download_with_progress(ckpt_url, 'base_checkpoint/vi_base.ckpt', desc='Mô hình Tiếng Việt Gốc (Base Checkpoint)')
download_with_progress(cfg_url, 'base_checkpoint/config.json', desc='Cấu hình Ngữ âm Tiếng Việt (Config)')
base_checkpoint_path = Path('base_checkpoint/vi_base.ckpt').resolve()
base_config_path = Path('base_checkpoint/config.json').resolve()
if not base_checkpoint_path.is_file() or base_checkpoint_path.stat().st_size < 1_000_000:
    raise FileNotFoundError(f'Checkpoint tiếng Việt chưa tải được hoặc không hợp lệ: {base_checkpoint_path}. Chạy lại ô Bước 1 khi có kết nối mạng.')
if not base_config_path.is_file() or base_config_path.stat().st_size == 0:
    raise FileNotFoundError(f'Config Piper chưa tải được hoặc rỗng: {base_config_path}. Chạy lại ô Bước 1 khi có kết nối mạng.')

ver_res = subprocess.run(
    [PIPER_PYTHON, '-c', 'import torch, piper_train, piper_phonemize; from piper_train.vits.monotonic_align import maximum_path; print("PIPER_TRAIN_READY", torch.__version__, "CUDA", torch.cuda.is_available())'],
    env=piper_env,
    capture_output=True, text=True
)
if ver_res.returncode != 0 or 'PIPER_TRAIN_READY' not in ver_res.stdout:
    raise RuntimeError(f'Piper dependencies are not ready in {PIPER_PYTHON}:\n{ver_res.stderr or ver_res.stdout}')
print(ver_res.stdout.strip(), flush=True)
if 'CUDA True' not in ver_res.stdout:
    raise RuntimeError('PyTorch không nhận được GPU CUDA. Hãy chọn T4 GPU trong Runtime type rồi chạy lại.')
print('\n======================================================================', flush=True)
print(f'🎉 MÔI TRƯỜNG PYTHON 3.11 ĐÃ SẴN SÀNG: {PIPER_PYTHON}', flush=True)
print('Hãy chạy Bước 2 để ghi ứng dụng, sau đó chạy Bước 3 để mở Gradio.', flush=True)
print(f'Log cài đặt: {SETUP_LOG}', flush=True)
print('======================================================================\n', flush=True)


In [ ]:
%%writefile /content/piper_gradio_app.py
# %%writefile /content/piper_gradio_app.py
import os, sys, shutil, subprocess, json, zipfile, glob, re, time, traceback, tempfile, wave, queue, threading, pathlib, random
from pathlib import Path

print('Gradio app: importing PyTorch...', flush=True)
import torch
print('Gradio app: importing pandas...', flush=True)
import pandas as pd
print('Gradio app: importing Gradio...', flush=True)
import gradio as gr
print('Gradio app: imports complete; building UI...', flush=True)

try:
    import piper_phonemize
except ImportError as e:
    raise RuntimeError('Thiếu piper-phonemize trong Python 3.11. Hãy chạy lại ô Bước 1.') from e

PIPER_SRC_DIR = os.path.abspath('piper/src/python')
if os.path.exists(PIPER_SRC_DIR) and PIPER_SRC_DIR not in sys.path:
    sys.path.insert(0, PIPER_SRC_DIR)
os.environ['PYTHONPATH'] = PIPER_SRC_DIR

BASE_DIR = Path('/content/piper_workspace').resolve()
DATASET_DIR = BASE_DIR / 'dataset'
WAVS_DIR = DATASET_DIR / 'wav'
TRAIN_DIR = BASE_DIR / 'training_dir'
OUTPUT_DIR = BASE_DIR / 'output'
DRIVE_MYDRIVE = Path('/content/drive/MyDrive')
DEFAULT_BACKUP_DIR = DRIVE_MYDRIVE / 'Piper_Backup' / 'checkpoints'

for d in [BASE_DIR, DATASET_DIR, WAVS_DIR, TRAIN_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

TRAIN_PROCESS = None
LAST_TRAINED_CHECKPOINT = None

def checkpoint_sort_key(checkpoint_path):
    checkpoint_path = str(checkpoint_path)
    match = re.search(r'epoch=(\d+).*?step=(\d+)', Path(checkpoint_path).name)
    if match:
        return (1, int(match.group(1)), int(match.group(2)), os.path.getmtime(checkpoint_path))
    return (0, -1, -1, os.path.getmtime(checkpoint_path))

def save_latest_checkpoint_to_drive(voice_name, custom_backup_dir=None):
    global LAST_TRAINED_CHECKPOINT
    if not LAST_TRAINED_CHECKPOINT or not Path(LAST_TRAINED_CHECKPOINT).is_file():
        # Thử tìm checkpoint mới nhất trong training_dir nếu biến tạm bị reset
        safe_voice_id = re.sub(r'[^a-zA-Z0-9_-]', '_', voice_name or 'my_voice').lower().strip('_') or 'my_voice'
        candidates = glob.glob(str(TRAIN_DIR / safe_voice_id / 'lightning_logs' / '**' / '*.ckpt'), recursive=True)
        if candidates:
            LAST_TRAINED_CHECKPOINT = max(candidates, key=checkpoint_sort_key)
        else:
            return '❌ Chưa có checkpoint từ lần huấn luyện vừa hoàn thành. Hãy train xong rồi bấm lưu.'

    drive_root = DRIVE_MYDRIVE
    if not drive_root.is_dir():
        return '❌ Chưa thấy Google Drive được mount. Hãy chạy lệnh mount Google Drive trong Colab rồi thử lại:\nfrom google.colab import drive; drive.mount(\'/content/drive\')'

    checkpoint_path = Path(LAST_TRAINED_CHECKPOINT).resolve()
    safe_voice_id = re.sub(r'[^a-zA-Z0-9_-]', '_', voice_name or 'my_voice').lower().strip('_') or 'my_voice'
    
    if custom_backup_dir and str(custom_backup_dir).strip():
        # Luon tao subfolder <voice_id> ben trong thu muc duoc chi dinh
        backup_dir = Path(str(custom_backup_dir).strip()).resolve() / safe_voice_id
    else:
        backup_dir = DEFAULT_BACKUP_DIR / safe_voice_id

    destination = backup_dir / checkpoint_path.name
    temporary_destination = destination.with_name(destination.name + '.uploading')

    try:
        if checkpoint_path == destination.resolve():
            return f'✅ Checkpoint đã nằm trong thư mục backup: {destination}'
        backup_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(checkpoint_path, temporary_destination)
        os.replace(temporary_destination, destination)
        return (
            f'✅ ĐÃ LƯU CHECKPOINT THÀNH CÔNG VÀO GOOGLE DRIVE!\n'
            f'📁 Thư mục Voice ID: {safe_voice_id}\n'
            f'📂 Đường dẫn đầy đủ: {destination}\n'
            f'📦 File: {checkpoint_path.name}\n'
            f'📊 Dung lượng: {destination.stat().st_size / (1024 * 1024):.1f} MB'
        )
    except Exception as e:
        if temporary_destination.exists():
            temporary_destination.unlink()
        return f'❌ Không thể sao lưu checkpoint lên Drive: {e}'

def validate_wav_file(wav_path):
    try:
        with wave.open(str(wav_path), 'rb') as audio:
            if audio.getnframes() <= 0 or audio.getframerate() != 22050 or audio.getnchannels() != 1 or audio.getsampwidth() != 2:
                return False, 'cần WAV mono 22050 Hz, PCM 16-bit và có âm thanh'
        return True, ''
    except (wave.Error, EOFError) as e:
        return False, f'File không phải WAV hợp lệ: {e}'
    except Exception as e:
        return False, f'Lỗi đọc audio: {e}'

def load_and_prepare_dataset(
    source_type,
    drive_dataset_path,
    upload_zip_obj,
    voice_name_hint,
    do_shuffle,
    num_samples_to_train
):
    """
    Nạp dataset từ Google Drive (ZIP hoặc Folder) hoặc File Upload.
    Hỗ trợ Shuffle ngẫu nhiên và chọn N câu để train mà KHÔNG ảnh hưởng dataset gốc.
    """
    staging_dir = None
    try:
        staging_dir = Path(tempfile.mkdtemp(prefix='dataset_stage_', dir=BASE_DIR))
        staged_wavs = staging_dir / 'wav'
        staged_wavs.mkdir(parents=True, exist_ok=True)
        meta_file = staging_dir / 'metadata.csv'

        inferred_voice_name = voice_name_hint

        # ── 1. Thu thập dữ liệu từ nguồn ──
        if source_type == 'drive':
            drive_path_str = (drive_dataset_path or '').strip()
            if not drive_path_str:
                return '❌ Vui lòng nhập đường dẫn Dataset trên Google Drive.', 'Chưa nhập đường dẫn', [], voice_name_hint, 0
            
            src_path = Path(drive_path_str).expanduser().resolve()
            if not src_path.exists():
                return f'❌ Đường dẫn không tồn tại trên Colab/Drive: {src_path}\nHãy chắc chắn bạn đã mount Drive.', 'Đường dẫn không tồn tại', [], voice_name_hint, 0

            if src_path.is_file() and (src_path.suffix.lower() == '.zip' or zipfile.is_zipfile(src_path)):
                # Nạp từ file ZIP trên Google Drive
                if not inferred_voice_name or inferred_voice_name == 'giong_moi':
                    bname = src_path.stem
                    if bname.endswith('_dataset'): bname = bname[:-len('_dataset')]
                    inferred_voice_name = re.sub(r'[^a-zA-Z0-9_-]', '_', bname).lower().strip('_') or 'giong_moi'

                with zipfile.ZipFile(src_path, 'r') as z:
                    meta_in_zip = None
                    wav_members = {}
                    for item in z.infolist():
                        if item.is_dir(): continue
                        fn = item.filename.replace('\\', '/')
                        b_name = Path(fn).name
                        if b_name.lower() == 'metadata.csv':
                            meta_in_zip = item
                        elif b_name.lower().endswith('.wav'):
                            cid = Path(b_name).stem
                            wav_members[cid] = (b_name, item)

                    if not meta_in_zip:
                        raise ValueError(f'File ZIP trên Drive {src_path.name} không chứa metadata.csv.')

                    with z.open(meta_in_zip) as s_f, open(meta_file, 'wb') as d_f:
                        shutil.copyfileobj(s_f, d_f)

                    # Tạm thời giải nén wavs vào staging
                    for b_name, item in wav_members.values():
                        with z.open(item) as s_f, open(staged_wavs / b_name, 'wb') as d_f:
                            shutil.copyfileobj(s_f, d_f)

            elif src_path.is_dir():
                # Nạp từ Thư mục giải nén sẵn trên Google Drive
                if not inferred_voice_name or inferred_voice_name == 'giong_moi':
                    inferred_voice_name = re.sub(r'[^a-zA-Z0-9_-]', '_', src_path.name).lower().strip('_') or 'giong_moi'

                # Tìm metadata.csv
                found_meta = None
                for candidate in [src_path / 'metadata.csv', src_path / 'dataset' / 'metadata.csv']:
                    if candidate.is_file():
                        found_meta = candidate
                        break
                if not found_meta:
                    all_csvs = list(src_path.glob('*.csv')) + list(src_path.glob('*/*.csv'))
                    if all_csvs:
                        found_meta = all_csvs[0]
                    else:
                        raise ValueError(f'Không tìm thấy metadata.csv trong thư mục {src_path}.')

                shutil.copy2(found_meta, meta_file)

                # Tìm wavs
                src_wav_dir = None
                for candidate_dir in [src_path / 'wav', src_path / 'wavs', src_path]:
                    if candidate_dir.is_dir() and list(candidate_dir.glob('*.wav')):
                        src_wav_dir = candidate_dir
                        break

                if not src_wav_dir:
                    raise ValueError(f'Không tìm thấy file .wav nào trong thư mục {src_path}.')

                # Copy các file wav từ Drive sang staging trên Colab
                for wav_file in src_wav_dir.glob('*.wav'):
                    shutil.copy2(wav_file, staged_wavs / wav_file.name)
            else:
                return f'❌ Đường dẫn {src_path} không phải file ZIP hoặc thư mục hợp lệ.', 'Sai định dạng', [], voice_name_hint, 0

        else:
            # Nạp từ file ZIP tải lên từ máy tính
            if not upload_zip_obj:
                return '❌ Chưa có file ZIP được tải lên.', 'Chưa có file', [], voice_name_hint, 0
            
            if isinstance(upload_zip_obj, dict):
                zip_path = upload_zip_obj.get('path') or upload_zip_obj.get('name')
            elif isinstance(upload_zip_obj, (str, os.PathLike)):
                zip_path = upload_zip_obj
            else:
                zip_path = getattr(upload_zip_obj, 'path', None) or getattr(upload_zip_obj, 'name', None)

            if not zip_path or not Path(zip_path).is_file():
                return '❌ Không đọc được file ZIP tải lên. Hãy thử tải lại.', 'Lỗi file', [], voice_name_hint, 0

            zip_path = Path(zip_path)
            if not zipfile.is_zipfile(zip_path):
                return '❌ File tải lên không đúng định dạng ZIP.', 'Lỗi định dạng', [], voice_name_hint, 0

            if not inferred_voice_name or inferred_voice_name == 'giong_moi':
                bname = zip_path.stem
                if bname.endswith('_dataset'): bname = bname[:-len('_dataset')]
                inferred_voice_name = re.sub(r'[^a-zA-Z0-9_-]', '_', bname).lower().strip('_') or 'giong_moi'

            with zipfile.ZipFile(zip_path, 'r') as z:
                meta_item = None
                wav_members = {}
                for item in z.infolist():
                    if item.is_dir(): continue
                    fn = item.filename.replace('\\', '/')
                    b_name = Path(fn).name
                    if b_name.lower() == 'metadata.csv':
                        meta_item = item
                    elif b_name.lower().endswith('.wav'):
                        cid = Path(b_name).stem
                        wav_members[cid] = (b_name, item)

                if not meta_item:
                    raise ValueError('File ZIP tải lên không chứa metadata.csv.')

                with z.open(meta_item) as s_f, open(meta_file, 'wb') as d_f:
                    shutil.copyfileobj(s_f, d_f)

                for b_name, item in wav_members.values():
                    with z.open(item) as s_f, open(staged_wavs / b_name, 'wb') as d_f:
                        shutil.copyfileobj(s_f, d_f)

        # ── 2. Đọc và kiểm tra metadata.csv ──
        all_rows = []
        clip_ids = set()
        with open(meta_file, 'r', encoding='utf-8-sig') as f:
            for line_no, line in enumerate(f, start=1):
                line = line.strip()
                if not line: continue
                if '|' not in line:
                    raise ValueError(f'metadata.csv dòng {line_no} thiếu dấu | phân cách ID và văn bản.')
                cid, txt = line.split('|', 1)
                cid, txt = cid.strip(), txt.strip()
                if not cid or not txt:
                    continue
                if cid in clip_ids:
                    continue  # Bỏ qua dòng trùng ID
                clip_ids.add(cid)
                all_rows.append([cid, txt])

        if not all_rows:
            raise ValueError('metadata.csv không chứa câu thoại nào hợp lệ.')

        # Kiểm tra file WAV tương ứng
        available_wavs = {p.stem: p for p in staged_wavs.glob('*.wav')}
        valid_rows = []
        for cid, txt in all_rows:
            if cid in available_wavs:
                is_valid, msg = validate_wav_file(available_wavs[cid])
                if is_valid:
                    valid_rows.append([cid, txt])
                else:
                    print(f'Bỏ qua clip {cid}: {msg}')

        total_original_count = len(valid_rows)
        if total_original_count == 0:
            raise ValueError('Không có file WAV nào đạt chuẩn mono 22050 Hz PCM 16-bit khớp với metadata.')

        # ── 3. Random Shuffle (Xáo trộn ngẫu nhiên) ──
        processed_rows = list(valid_rows)
        if do_shuffle:
            # Dùng seed ngẫu nhiên theo thời gian để mỗi lần nạp/bấm shuffle sẽ ra thứ tự mới
            random.seed(int(time.time() * 1000) % 2147483647)
            random.shuffle(processed_rows)

        # ── 4. Chọn số lượng câu muốn train (Subsampling) ──
        num_to_take = total_original_count
        if num_samples_to_train and int(num_samples_to_train) > 0:
            num_to_take = min(int(num_samples_to_train), total_original_count)

        selected_rows = processed_rows[:num_to_take]
        selected_clip_ids = {row[0] for row in selected_rows}

        # ── 5. Triển khai vào thư mục DATASET_DIR chính thức ──
        if DATASET_DIR.exists():
            shutil.rmtree(DATASET_DIR)
        DATASET_DIR.mkdir(parents=True, exist_ok=True)
        final_wavs_dir = DATASET_DIR / 'wav'
        final_wavs_dir.mkdir(parents=True, exist_ok=True)

        # Ghi metadata.csv mới với đúng danh sách đã chọn
        final_meta = DATASET_DIR / 'metadata.csv'
        with open(final_meta, 'w', encoding='utf-8') as f:
            for cid, txt in selected_rows:
                f.write(f'{cid}|{txt}\n')

        # Chỉ copy các file WAV được chọn vào DATASET_DIR
        for cid in selected_clip_ids:
            wav_file = available_wavs.get(cid)
            if wav_file and wav_file.exists():
                shutil.copy2(wav_file, final_wavs_dir / wav_file.name)

        shuffle_msg = '🔀 Đã xáo trộn ngẫu nhiên' if do_shuffle else 'Giữ nguyên thứ tự gốc'
        summary = (
            f'✅ NẠP DATASET THÀNH CÔNG!\n'
            f'- Tổng số câu thoại trong dataset gốc: {total_original_count} câu.\n'
            f'- Trạng thái xáo trộn: {shuffle_msg}.\n'
            f'- Số câu được chọn để huấn luyện: {len(selected_rows)} / {total_original_count} câu.\n'
            f'- Thư mục dataset huấn luyện: {DATASET_DIR} (Dataset gốc trên Drive hoàn toàn không bị ảnh hưởng).'
        )
        status = f'Sẵn sàng huấn luyện: {len(selected_rows)} câu thoại'
        
        return summary, status, selected_rows[:100], inferred_voice_name, total_original_count

    except Exception as e:
        traceback.print_exc()
        return f'❌ Lỗi khi nạp dataset: {str(e)}', 'Lỗi xử lý', [], voice_name_hint, 0
    finally:
        if staging_dir is not None and staging_dir.exists():
            shutil.rmtree(staging_dir, ignore_errors=True)

def start_training(
    voice_name,
    checkpoint_mode,
    drive_checkpoint_path,
    epochs,
    batch_size,
    progress=gr.Progress()
):
    global TRAIN_PROCESS, LAST_TRAINED_CHECKPOINT
    try:
        v_id = re.sub(r'[^a-zA-Z0-9_-]', '_', voice_name or 'my_voice').lower().strip('_')
        if not v_id: v_id = 'my_voice'

        # Kiểm tra dataset đã được nạp chưa
        meta_file = DATASET_DIR / 'metadata.csv'
        wav_files = list(WAVS_DIR.glob('*.wav'))
        if not meta_file.exists() or not wav_files:
            yield '❌ Chưa có dữ liệu huấn luyện! Hãy nạp Dataset ở Bước 1 trước.', 'Chưa có dataset', ''
            return

        with open(meta_file, 'r', encoding='utf-8') as f:
            cues_count = len([line for line in f if line.strip()])

        # Xác định checkpoint bắt đầu
        checkpoint_path = None
        checkpoint_epoch = 0
        base_ckpt = Path('base_checkpoint/vi_base.ckpt').resolve()

        if checkpoint_mode == 'drive':
            ckpt_str = (drive_checkpoint_path or '').strip()
            if not ckpt_str:
                yield '❌ Bạn chọn dùng Checkpoint từ Google Drive nhưng chưa nhập đường dẫn.', 'Thiếu checkpoint', ''
                return
            cp = Path(ckpt_str).expanduser().resolve()
            if not cp.is_file() or cp.suffix.lower() != '.ckpt':
                yield f'❌ Không tìm thấy file checkpoint .ckpt hợp lệ tại: {cp}', 'Checkpoint không hợp lệ', ''
                return
            checkpoint_path = cp
            m = re.search(r'epoch=(\d+)', checkpoint_path.name)
            if m:
                checkpoint_epoch = int(m.group(1))
            else:
                try:
                    with torch.serialization.safe_globals([pathlib.PosixPath, pathlib.WindowsPath]):
                        ckpt_data = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
                    stored_epoch = ckpt_data.get('epoch') if isinstance(ckpt_data, dict) else None
                    if isinstance(stored_epoch, int) and stored_epoch >= 0:
                        checkpoint_epoch = stored_epoch
                    del ckpt_data
                except Exception as e:
                    yield f'❌ Không đọc được epoch từ checkpoint: {e}. Vui lòng đổi tên file theo định dạng epoch=N-step=M.ckpt.', 'Checkpoint lỗi', ''
                    return

        elif checkpoint_mode == 'base':
            if not base_ckpt.is_file():
                yield f'❌ Không tìm thấy checkpoint nền tiếng Việt: {base_ckpt}. Hãy chạy lại Bước 1 trong Colab để tải.', 'Thiếu base checkpoint', ''
                return
            checkpoint_path = base_ckpt
            checkpoint_epoch = 4769

        else:
            # Train from scratch
            checkpoint_path = None
            checkpoint_epoch = 0

        target_max_epochs = int(epochs) + checkpoint_epoch

        progress(0.1, desc=f'Đang tiền xử lý ngữ âm Tiếng Việt cho {cues_count} câu thoại...')
        yield f'⏳ Đang tiền xử lý ngữ âm (piper-phonemize vi) cho {cues_count} câu...', 'Tiền xử lý dataset', ''

        prep_cmd = [
            sys.executable, '-m', 'piper_train.preprocess',
            '--language', 'vi',
            '--input-dir', str(DATASET_DIR),
            '--output-dir', str(DATASET_DIR),
            '--dataset-format', 'ljspeech',
            '--sample-rate', '22050',
            '--single-speaker'
        ]
        prep_res = subprocess.run(prep_cmd, capture_output=True, text=True, env=os.environ)
        if prep_res.returncode != 0:
            yield f'❌ Lỗi khi tiền xử lý ngữ âm: {prep_res.stderr or prep_res.stdout}', 'Lỗi tiền xử lý', ''
            return

        progress(0.25, desc='Khởi tạo cấu hình Piper Training...')
        train_output_dir = TRAIN_DIR / v_id
        train_output_dir.mkdir(parents=True, exist_ok=True)

        train_cmd = [
            sys.executable, '-c',
            'import pathlib, runpy, sys, torch; '
            'torch.serialization.add_safe_globals([pathlib.PosixPath]); '
            'sys.argv = ["piper_train"] + sys.argv[1:]; '
            'runpy.run_module("piper_train", run_name="__main__")',
            '--dataset-dir', str(DATASET_DIR),
            '--default_root_dir', str(train_output_dir),
            '--accelerator', 'gpu' if torch.cuda.is_available() else 'cpu',
            '--devices', '1',
            '--logger', 'True',
            '--batch-size', str(int(batch_size)),
            '--max_epochs', str(target_max_epochs),
            '--checkpoint-epochs', '1',
            '--quality', 'medium'
        ]

        if checkpoint_path:
            train_cmd.extend(['--resume_from_checkpoint', str(checkpoint_path)])

        progress(0.35, desc=f'Bắt đầu huấn luyện mạng VITS ({epochs} epochs, {cues_count} câu thoại)...')
        print(f"Executing: {' '.join(train_cmd)}")

        train_env = os.environ.copy()
        train_env['WANDB_MODE'] = 'disabled'
        train_env['WANDB_DISABLED'] = 'true'
        train_env['PYTHONUNBUFFERED'] = '1'

        proc = subprocess.Popen(
            train_cmd,
            cwd=str(train_output_dir),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
            env=train_env
        )
        TRAIN_PROCESS = proc

        logs = []
        current_epoch = None
        epoch_started_at = None
        epoch_durations = []
        training_started_at = time.monotonic()
        last_ui_update = 0.0
        output_queue = queue.Queue()

        def read_output():
            pending = ''
            try:
                while True:
                    char = proc.stdout.read(1)
                    if not char: break
                    if char in ('\r', '\n'):
                        if pending.strip():
                            output_queue.put(pending.strip())
                        pending = ''
                    else:
                        pending += char
                if pending.strip():
                    output_queue.put(pending.strip())
            finally:
                output_queue.put(None)

        threading.Thread(target=read_output, daemon=True).start()
        output_done = False

        while not output_done or proc.poll() is None:
            try:
                clean_line = output_queue.get(timeout=1.0)
                if clean_line is None:
                    output_done = True
                else:
                    logs.append(clean_line)
                    if len(logs) > 100: logs.pop(0)

                    match = re.search(r'Epoch\s+(\d+)', clean_line)
                    if match:
                        observed_epoch = int(match.group(1))
                        now = time.monotonic()
                        if current_epoch is not None and observed_epoch != current_epoch and epoch_started_at is not None:
                            dur = now - epoch_started_at
                            if dur > 0:
                                epoch_durations.append(dur)
                                epoch_durations = epoch_durations[-5:]
                        if observed_epoch != current_epoch:
                            current_epoch = observed_epoch
                            epoch_started_at = now
                        
                        completed_epochs = max(0, min(int(epochs), current_epoch - checkpoint_epoch))
                        pct = min(0.95, 0.35 + 0.60 * (completed_epochs / max(1, int(epochs))))
                        progress(pct, desc=f'Đang huấn luyện ({cues_count} câu): Epoch {max(1, completed_epochs)}/{epochs}...')
            except queue.Empty:
                pass

            now = time.monotonic()
            if now - last_ui_update >= 1.0:
                elapsed_sec = int(now - training_started_at)
                elapsed_str = f'{elapsed_sec // 3600:02d}:{(elapsed_sec % 3600) // 60:02d}:{elapsed_sec % 60:02d}'
                if current_epoch is None:
                    epoch_text = 'Đang chờ log epoch đầu tiên...'
                    eta_text = 'đang tính toán...'
                else:
                    completed_epochs = max(0, min(int(epochs), current_epoch - checkpoint_epoch))
                    epoch_text = f'Epoch {max(1, completed_epochs)}/{epochs}'
                    if epoch_durations:
                        avg_ep = sum(epoch_durations) / len(epoch_durations)
                        rem_epochs = max(0, int(epochs) - completed_epochs)
                        cur_rem = max(0.0, avg_ep - (now - epoch_started_at))
                        eta_sec = int(cur_rem + max(0, rem_epochs - 1) * avg_ep)
                        eta_text = f'~{eta_sec // 3600:02d}:{(eta_sec % 3600) // 60:02d}:{eta_sec % 60:02d}'
                    else:
                        eta_text = 'đang thu thập dữ liệu'

                live_status = f'⏳ Đang huấn luyện — {epoch_text} | {cues_count} câu | đã chạy {elapsed_str} | ETA {eta_text}'
                live_logs = '\n'.join(logs[-45:]) or 'Đang nhận log từ Piper...'
                yield live_status, 'Đang huấn luyện', live_logs
                last_ui_update = now

        proc.wait()
        if proc.returncode != 0:
            err = '\n'.join(logs[-40:]) or f'Process exit code {proc.returncode}.'
            yield f'❌ Huấn luyện thất bại:\n{err}', 'Lỗi huấn luyện', '\n'.join(logs[-40:])
            return

        run_checkpoints = glob.glob(str(train_output_dir / 'lightning_logs' / '**' / '*.ckpt'), recursive=True)
        if run_checkpoints:
            LAST_TRAINED_CHECKPOINT = max(run_checkpoints, key=checkpoint_sort_key)
            print(f'Checkpoint vừa huấn luyện thành công: {LAST_TRAINED_CHECKPOINT}')
        else:
            LAST_TRAINED_CHECKPOINT = None

        progress(1.0, desc='✅ Huấn luyện hoàn tất thành công!')
        success_msg = (
            f'🎉 HUẤN LUYỆN THÀNH CÔNG {epochs} EPOCHS!\n'
            f'- Giọng: {voice_name}\n'
            f'- Số câu đã train: {cues_count} câu\n'
            f'- Checkpoint mới nhất: {Path(LAST_TRAINED_CHECKPOINT).name if LAST_TRAINED_CHECKPOINT else "Đã lưu"}\n\n'
            f'👉 BƯỚC TIẾP THEO:\n'
            f'1. Bấm nút "💾 LƯU CHECKPOINT VÀO GOOGLE DRIVE" bên dưới để sao lưu lâu dài.\n'
            f'2. Hoặc sang Tab "2. Xuất File Mô Hình" để tải file .onnx và .onnx.json về máy tính!'
        )
        yield success_msg, 'Huấn luyện hoàn tất', '\n'.join(logs[-45:])

    except Exception as e:
        yield f'❌ Lỗi huấn luyện: {str(e)}\n{traceback.format_exc()}', 'Lỗi huấn luyện', ''

def export_onnx_model(voice_name, progress=gr.Progress()):
    try:
        v_id = re.sub(r'[^a-zA-Z0-9_-]', '_', voice_name or 'my_voice').lower().strip('_')
        if not v_id: v_id = 'my_voice'

        train_dir = TRAIN_DIR / v_id
        checkpoint_roots = [train_dir, DATASET_DIR / 'lightning_logs']
        ckpts = sorted({
            checkpoint
            for root in checkpoint_roots
            if root.exists()
            for checkpoint in glob.glob(str(root / '**/*.ckpt'), recursive=True)
        })

        if not ckpts:
            searched = ', '.join(str(root) for root in checkpoint_roots)
            return f'❌ Không tìm thấy checkpoint huấn luyện cho giọng "{voice_name}". Đã kiểm tra: {searched}.', None, None

        latest_ckpt = LAST_TRAINED_CHECKPOINT
        if not latest_ckpt or not Path(latest_ckpt).is_file():
            latest_ckpt = max(ckpts, key=checkpoint_sort_key)

        print(f'Exporting checkpoint: {latest_ckpt}')
        progress(0.4, desc=f'Đang export checkpoint {os.path.basename(latest_ckpt)} sang ONNX...')

        out_onnx = OUTPUT_DIR / f'{v_id}.onnx'
        out_json = OUTPUT_DIR / f'{v_id}.onnx.json'

        export_cmd = [
            sys.executable, '-c',
            'import pathlib, runpy, sys, torch; '
            'torch.serialization.add_safe_globals([pathlib.PosixPath]); '
            'sys.argv = ["piper_train.export_onnx"] + sys.argv[1:]; '
            'runpy.run_module("piper_train.export_onnx", run_name="__main__")',
            latest_ckpt,
            str(out_onnx)
        ]
        exp_res = subprocess.run(export_cmd, capture_output=True, text=True, env=os.environ)
        if exp_res.returncode != 0 or not out_onnx.exists():
            return f'❌ Lỗi xuất ONNX: {exp_res.stderr or exp_res.stdout}', None, None

        cfg_path = train_dir / 'config.json'
        if not cfg_path.exists():
            cfg_path = Path('base_checkpoint/config.json')
        if cfg_path.exists():
            shutil.copy(str(cfg_path), str(out_json))

        progress(1.0, desc='✅ Đã xuất mô hình ONNX thành công!')
        msg = (
            f'✅ XUẤT MÔ HÌNH THÀNH CÔNG!\n'
            f'- Nguồn: {latest_ckpt}\n'
            f'- File ONNX: {out_onnx.name} ({out_onnx.stat().st_size / (1024*1024):.1f} MB)\n'
            f'- File Cấu hình: {out_json.name}'
        )
        return msg, str(out_onnx), str(out_json)
    except Exception as e:
        return f'❌ Lỗi khi xuất ONNX: {str(e)}', None, None

# ── Xây dựng Giao diện Gradio ──
custom_css = '''
.gradio-container { max-width: 980px !important; margin: auto; font-family: system-ui, -apple-system, sans-serif; }
.main-btn { background: linear-gradient(135deg, #f59e0b, #ea580c) !important; color: #020617 !important; font-weight: 700 !important; }
.save-btn { background: linear-gradient(135deg, #10b981, #059669) !important; color: #ffffff !important; font-weight: 700 !important; }
.section-box { border: 1px solid #334155; border-radius: 8px; padding: 12px; margin-bottom: 12px; }
'''

with gr.Blocks(theme=gr.themes.Soft(), css=custom_css, title='Piper Voice Clone Studio (Pro)') as demo:
    gr.Markdown(
        '''
        # 🎙️ Piper Vietnamese Voice Training Studio (Advanced Edition)
        **Huấn luyện giọng nói tiếng Việt VITS siêu tốc trên Google Colab GPU T4.**
        *Hỗ trợ: Nạp dataset từ Google Drive, Random Shuffle, Subsampling (chọn số câu), Load/Save Checkpoint từ Drive.*
        '''
    )

    with gr.Tabs():
        # TAB 1: DATASET & TRAINING
        with gr.TabItem('1. Nạp Dataset & Huấn Luyện (Training)'):
            with gr.Group():
                gr.Markdown('### 📦 Bước 1: Thiết Lập Nguồn Dataset & Chọn Số Câu Train')
                with gr.Row():
                    source_type_radio = gr.Radio(
                        choices=[('Đường dẫn Google Drive (Khuyên dùng)', 'drive'), ('Tải file ZIP từ máy tính', 'upload')],
                        value='drive',
                        label='Chọn nguồn nạp Dataset'
                    )

                PRESET_DATASETS = [
                    ("📦 kevinha_number_247_dataset.zip (247 câu)", "/content/drive/MyDrive/Piper_Backup/kevinha_number_247_dataset.zip"),
                    ("📦 kevinha_tutien_819_dataset.zip (819 câu)", "/content/drive/MyDrive/Piper_Backup/kevinha_tutien_819_dataset.zip"),
                    ("📦 kevinha_voice_2625_dataset.zip (2625 câu)", "/content/drive/MyDrive/Piper_Backup/kevinha_voice_2625_dataset.zip"),
                    ("✏️ Nhập đường dẫn tùy chỉnh khác...", "custom"),
                ]

                dataset_dropdown = gr.Dropdown(
                    label='🎯 Danh sách Dataset có sẵn trên Google Drive (chọn để dùng path tương ứng)',
                    choices=PRESET_DATASETS,
                    value=PRESET_DATASETS[0][1],
                    info='Chọn dataset để tự động điền đường dẫn và đặt tên Voice ID tương ứng'
                )

                drive_dataset_input = gr.Textbox(
                    label='Đường dẫn Dataset trên Google Drive (File .zip hoặc Thư mục)',
                    placeholder='/content/drive/MyDrive/Piper_Backup/...',
                    value=PRESET_DATASETS[0][1]
                )

                upload_dataset_input = gr.File(
                    label='Tải file dataset .zip từ máy tính (nếu không dùng Google Drive)',
                    file_types=['.zip'],
                    type='filepath',
                    visible=False
                )

                with gr.Row():
                    voice_name_input = gr.Textbox(
                        label='Tên Giọng / Voice ID',
                        value='kevinha_number_247',
                        placeholder='VD: kevinha_number_247, hoang_nam'
                    )
                    do_shuffle_check = gr.Checkbox(
                        label='🔀 Xáo trộn ngẫu nhiên câu thoại (Random Shuffle)',
                        value=True,
                        info='Xáo trộn trong Colab, không ảnh hưởng dataset gốc trên Drive'
                    )

                def on_dataset_dropdown_change(selected_val):
                    if selected_val and selected_val != "custom":
                        stem = Path(selected_val).stem
                        if stem.endswith('_dataset'):
                            stem = stem[:-len('_dataset')]
                        clean_name = re.sub(r'[^a-zA-Z0-9_-]', '_', stem).lower().strip('_')
                        return gr.update(value=selected_val), gr.update(value=clean_name)
                    elif selected_val == "custom":
                        return gr.update(value="/content/drive/MyDrive/"), gr.update()
                    return gr.update(), gr.update()

                dataset_dropdown.change(
                    fn=on_dataset_dropdown_change,
                    inputs=[dataset_dropdown],
                    outputs=[drive_dataset_input, voice_name_input]
                )

                def on_source_type_change(val):
                    is_drive = (val == 'drive')
                    return (
                        gr.update(visible=is_drive),
                        gr.update(visible=is_drive),
                        gr.update(visible=not is_drive)
                    )

                source_type_radio.change(
                    fn=on_source_type_change,
                    inputs=[source_type_radio],
                    outputs=[dataset_dropdown, drive_dataset_input, upload_dataset_input]
                )

                with gr.Row():
                    num_samples_input = gr.Number(
                        label='🎯 Số lượng câu muốn train (Nhập số câu, hoặc 0 để train toàn bộ dataset)',
                        value=0,
                        precision=0,
                        info='Hệ thống sẽ chỉ lấy đúng số lượng câu này sau khi shuffle'
                    )

                btn_load_dataset = gr.Button('🔍 KIỂM TRA & NẠP DỮ LIỆU', variant='secondary')
                dataset_summary_box = gr.Markdown('Chờ nạp dữ liệu...')
                dataset_status_box = gr.Textbox(label='Trạng thái dữ liệu', interactive=False)
                dataset_preview_table = gr.Dataframe(
                    headers=['ID', 'Văn bản (Tiếng Việt)'],
                    label='Xem trước các câu thoại được chọn',
                    interactive=False
                )

                btn_load_dataset.click(
                    fn=load_and_prepare_dataset,
                    inputs=[source_type_radio, drive_dataset_input, upload_dataset_input, voice_name_input, do_shuffle_check, num_samples_input],
                    outputs=[dataset_summary_box, dataset_status_box, dataset_preview_table, voice_name_input]
                )

            with gr.Group():
                gr.Markdown('### ⚙️ Bước 2: Chọn Checkpoint & Cấu Hình Huấn Luyện')
                with gr.Row():
                    checkpoint_mode_radio = gr.Radio(
                        choices=[
                            ('Fine-tune từ Checkpoint Tiếng Việt Gốc (vi_base.ckpt)', 'base'),
                            ('Tiếp tục từ Checkpoint trên Google Drive (.ckpt)', 'drive'),
                            ('Train từ đầu (Từ scratch - không khuyến nghị)', 'scratch')
                        ],
                        value='base',
                        label='Điểm bắt đầu huấn luyện'
                    )

                drive_checkpoint_input = gr.Textbox(
                    label='Đường dẫn file Checkpoint .ckpt trên Google Drive',
                    placeholder='/content/drive/MyDrive/Piper_Backup/checkpoints/my_voice/epoch=4793-step=922412.ckpt',
                    visible=False
                )

                checkpoint_mode_radio.change(
                    fn=lambda m: gr.update(visible=(m == 'drive')),
                    inputs=[checkpoint_mode_radio],
                    outputs=[drive_checkpoint_input]
                )

                with gr.Row():
                    epochs_slider = gr.Slider(minimum=5, maximum=100, value=25, step=5, label='Số Epochs (Khuyên dùng: 20-30 cho dataset ngắn)')
                    batch_slider = gr.Slider(minimum=4, maximum=32, value=16, step=4, label='Batch Size (GPU T4: 16 là tối ưu)')

                btn_start_train = gr.Button('🚀 BẮT ĐẦU HUẤN LUYỆN (TRAIN MODEL)', elem_classes='main-btn', size='lg')
                train_output_status = gr.Textbox(label='Trạng Thái Huấn Luyện', interactive=False)
                train_log_output = gr.Textbox(label='Nhật ký huấn luyện (cập nhật trực tiếp)', lines=16, max_lines=28, interactive=False)

                btn_start_train.click(
                    fn=start_training,
                    inputs=[voice_name_input, checkpoint_mode_radio, drive_checkpoint_input, epochs_slider, batch_slider],
                    outputs=[train_output_status, dataset_status_box, train_log_output]
                )

            with gr.Group():
                gr.Markdown('### 💾 Bước 3: Sao Lưu Checkpoint Mới Vào Google Drive')
                with gr.Row():
                    custom_backup_dir_input = gr.Textbox(
                        label='Thư mục lưu Checkpoint trên Google Drive',
                        value=str(DEFAULT_BACKUP_DIR),
                        placeholder='/content/drive/MyDrive/Piper_Backup/checkpoints',
                        info='Hệ thống sẽ tự động tạo thư mục con theo Tên giọng / Voice ID bên trong (VD: .../checkpoints/<voice_id>/checkpoint.ckpt)'
                    )
                btn_save_checkpoint = gr.Button('💾 LƯU CHECKPOINT VỪA TRAIN VÀO GOOGLE DRIVE', elem_classes='save-btn', size='lg')
                save_checkpoint_status = gr.Textbox(label='Kết quả sao lưu', interactive=False)

                btn_save_checkpoint.click(
                    fn=save_latest_checkpoint_to_drive,
                    inputs=[voice_name_input, custom_backup_dir_input],
                    outputs=[save_checkpoint_status]
                )

        # TAB 2: EXPORT ONNX
        with gr.TabItem('2. Xuất File Mô Hình (.onnx & .onnx.json)'):
            gr.Markdown(
                '''
                ### ⚡ Xuất mô hình ONNX để nạp vào Piper TTS Studio
                Sau khi huấn luyện hoàn tất (hoặc đạt số epoch mong muốn), nhấn nút bên dưới để xuất:
                - `<ten_giong>.onnx`: Trọng số nơ-ron mạng VITS
                - `<ten_giong>.onnx.json`: Cấu hình ngữ âm & mẫu âm học
                '''
            )
            btn_export = gr.Button('⚡ XUẤT FILE .ONNX & .ONNX.JSON', elem_classes='main-btn', size='lg')
            export_status = gr.Markdown('Chờ xuất file...')
            with gr.Row():
                file_onnx_out = gr.File(label='File Mô Hình (.onnx)')
                file_json_out = gr.File(label='File Cấu Hình (.onnx.json)')

            btn_export.click(
                fn=export_onnx_model,
                inputs=[voice_name_input],
                outputs=[export_status, file_onnx_out, file_json_out]
            )

print('Gradio app: UI built; starting local server and public tunnel...', flush=True)
demo.queue().launch(share=True, debug=False)



In [ ]:
# @title 🚀 Bước 3: Khởi Chạy Gradio trong Python 3.11 { display-mode: "form" }
import os, subprocess, sys
PIPER_PYTHON = '/content/piper_py311/bin/python'
if not os.path.isfile(PIPER_PYTHON):
    raise FileNotFoundError(f'Không tìm thấy Python môi trường Piper: {PIPER_PYTHON}. Hãy chạy lại Bước 1.')
if not os.path.isfile('/content/piper_gradio_app.py'):
    raise FileNotFoundError('Không tìm thấy /content/piper_gradio_app.py. Hãy chạy lại Bước 2 để ghi ứng dụng.')
app_env = os.environ.copy()
piper_src = os.path.abspath('piper/src/python')
app_env['PYTHONPATH'] = piper_src
print('Đang khởi chạy ứng dụng Gradio bằng Python 3.11; log sẽ hiện bên dưới.', flush=True)
process = subprocess.Popen(
    [PIPER_PYTHON, '-u', '/content/piper_gradio_app.py'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=app_env
)
if process.stdout is None:
    raise RuntimeError('Không đọc được log từ tiến trình Gradio.')
with open('/content/piper_gradio.log', 'w', encoding='utf-8') as app_log:
    for line in process.stdout:
        print(line, end='', flush=True)
        app_log.write(line)
        app_log.flush()
return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, process.args)


In [ ]:
# @title 💾 (Dự phòng) Sao Lưu Checkpoint Mới Nhất Vào Google Drive Bằng Code { display-mode: "form" }
# @markdown **Ghi chú:** Trên giao diện Web Gradio (Bước 3) đã có sẵn nút bấm tự động lưu checkpoint vào thư mục theo Voice ID.
# @markdown Ô này dành cho trường hợp bạn muốn chạy sao lưu thủ công bằng script Colab.

voice_id = "giong_moi" # @param {type:"string"}
custom_backup_root = "/content/drive/MyDrive/Piper_Backup/checkpoints" # @param {type:"string"}

import os, glob, shutil, re
from pathlib import Path

TRAIN_DIR = Path('/content/piper_workspace/training_dir')
BACKUP_ROOT = Path(custom_backup_root.strip() if custom_backup_root.strip() else '/content/drive/MyDrive/Piper_Backup/checkpoints')

# Tìm tất cả checkpoints trong thư mục huấn luyện
ckpts = glob.glob(str(TRAIN_DIR / '**/*.ckpt'), recursive=True)
if not ckpts:
    print('❌ Không tìm thấy checkpoint nào trong /content/piper_workspace/training_dir.')
else:
    def get_sort_key(p):
        m = re.search(r'epoch=(\d+).*?step=(\d+)', Path(p).name)
        if m: return (1, int(m.group(1)), int(m.group(2)))
        return (0, 0, os.path.getmtime(p))

    latest_ckpt = max(ckpts, key=get_sort_key)
    
    # Xác định Voice ID: ưu tiên nhập từ form, nếu để mặc định/rỗng thì tự nhận diện từ thư mục huấn luyện
    safe_voice_id = re.sub(r'[^a-zA-Z0-9_-]', '_', voice_id or '').lower().strip('_')
    if not safe_voice_id or safe_voice_id == 'giong_moi':
        try:
            rel_parts = Path(latest_ckpt).relative_to(TRAIN_DIR).parts
            if rel_parts:
                safe_voice_id = rel_parts[0]
        except Exception:
            safe_voice_id = safe_voice_id or 'giong_moi'

    # Tạo thư mục riêng theo Voice ID
    target_backup_dir = BACKUP_ROOT / safe_voice_id
    target_backup_dir.mkdir(parents=True, exist_ok=True)

    dest_path = target_backup_dir / Path(latest_ckpt).name
    if Path(latest_ckpt).resolve() == dest_path.resolve():
        print(f'✅ Checkpoint đã nằm trong thư mục backup: {dest_path}')
    else:
        tmp_dest = dest_path.with_name(dest_path.name + '.uploading')
        shutil.copy2(latest_ckpt, tmp_dest)
        os.replace(tmp_dest, dest_path)
        print('✅ ĐÃ SAO LƯU CHECKPOINT VÀO GOOGLE DRIVE THÀNH CÔNG!')
        print(f'📁 Thư mục Voice ID: {safe_voice_id}')
        print(f'📂 Đường dẫn đầy đủ: {dest_path}')
        print(f'📦 File: {Path(latest_ckpt).name}')
        print(f'📊 Kích thước: {dest_path.stat().st_size / (1024*1024):.1f} MB')
